In [ ]:
# TODO:
# 1. (done) diskcache + Annotator class + threaded parallel annotation
# 2. (done) §§3-5 are qwen3_nothink; optional qwen3_think training+comparison is in §9
# 3. Suggest particular datasets for annotation 
#      PubMed 20k RCT https://huggingface.co/datasets/armanc/pubmed-rct20k
#      Invoices: https://huggingface.co/datasets/Voxel51/scanned_receipts
#      Legal IE: https://huggingface.co/datasets/theatticusproject/cuad
#      CV dataset: https://huggingface.co/datasets/sandeeppanem/resume-json-extraction-5k

# Lab: From LLM Annotation to a Tiny Fine-Tuned Extractor

In this session we walk through a complete **knowledge-distillation-style** pipeline on a small scale:

1. Take a small **unlabeled** corpus of news headlines.
2. Use **Gemini 3 Flash** via **OpenRouter API** as a strong *teacher* model to produce **structured JSON annotations** — named entities grouped into four categories.
3. Fine-tune a very small *student* model — **Qwen3 non-thinking** — with **LoRA** to reproduce the teacher's annotations.
4. Optionally run a second **Qwen3 thinking** regime in §9 and compare it against the non-thinking fine-tuned model.
4. Evaluate how well the tiny student mimics the huge teacher, compared with its zero-shot baseline.

**Why this task?** Structured information extraction is a sweet spot:
- the output is short and machine-checkable (it's just JSON),
- we can compute entity-level Precision / Recall / F1 automatically,
- LLMs are good at it zero-shot, so they are credible annotators,
- small models can learn it well from a few hundred examples.

**Hardware.** Anything with ~6 GB of GPU memory will do (a free Colab T4 is plenty). CPU-only works for inference but training will be slow.

**API key.** You need an OpenRouter API key (get one at <https://openrouter.ai/keys>). Paste it into the notebook cell below as `OPENROUTER_API_KEY`.


In [1]:
import os

os.environ["OPENROUTER_API_KEY"] = "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"

## 0. Install dependencies

In [2]:
%pip install -q --upgrade "transformers>=5.5" "trl>=0.20" "peft>=0.14" "accelerate>=1.0" \
    "datasets>=2.20" "openai>=1.0" "pydantic>=2.0" diskcache tqdm "torchao>=0.17.0" "torch>=2.10.0" torchvision

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, random, re
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import torch
from datasets import Dataset, load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("CUDA bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

/home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
CUDA bf16 supported: True


## 1. Prepare a small unlabeled corpus

We sample a couple of hundred headlines from the public **AG News** dataset. In a real project this is where you would plug in *your own* unlabeled texts (support tickets, chat messages, clinical notes, scientific abstracts, …).

In [4]:
raw = load_dataset("ag_news", split="train")
rng = random.Random(SEED)
pool = rng.sample(range(len(raw)), 1000)
headlines = [raw[i]["text"].strip() for i in pool]
headlines = [h for h in headlines if 30 < len(h) < 250][:400]

print(f"Corpus size: {len(headlines)} headlines\n")
for h in headlines[:5]:
    print("•", h)

Corpus size: 400 headlines

• Policeman 'saw fatal train crash' An off-duty policeman watched a train plough into a car on a level crossing  in Berkshire, killing six people.
• Silver finale for USA In the last event of the 2004 Olympic Games, the United States track team produced one last surprise. Meb Keflezighi, a native of Eritrea who moved to the United States as
• Polish Hostage Freed in Iraq Already in Warsaw  WARSAW (Reuters) - A Polish woman kidnapped in Iraq last  month has been freed and flown to Poland and said she was  treated well, raising hopes for other foreign hostages.
• Wall Street Journal to Start Saturday Issue he Wall Street Journal announced yesterday that it would begin publishing a Saturday issue starting in the fall of 2005. By following its business readers home for the weekend, the newspaper
• EU, Japan win WTO approval to impose duties on US The European Union, Japan, Brazil and five other countries won World Trade Organization approval to impose tariffs wo

## 2. Annotate with Gemini (the teacher)

We ask Gemini to extract four entity types and return a strict JSON object:

```json
{
  "persons":       ["..."],
  "organizations": ["..."],
  "locations":     ["..."],
  "events":        ["..."]
}
```

We use OpenRouter's OpenAI-compatible API with the `gemini-3-flash` model — fast enough for high-throughput annotation while keeping quality strong.

We enforce a strict JSON-only response and validate it with a Pydantic schema (`Entities`) before caching; this keeps the teacher outputs consistent for student training.

We also disable "thinking" (`thinking_budget=0`) because this is a short, structured task and we don't need extra reasoning tokens.

In [7]:
import json
import logging
import os
import threading
import diskcache as dc
from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI
from pydantic import BaseModel

log = logging.getLogger()


class Entities(BaseModel):
    persons: list[str]
    organizations: list[str]
    locations: list[str]
    events: list[str]


class Annotator:
    """
    OpenRouter-based entity extraction with diskcache and optional parallel annotation via threads.
    Each worker thread uses its own ``OpenAI`` client; the on-disk cache is shared and
    thread-safe for concurrent reads/writes.
    """

    SYSTEM_PROMPT = (
        "You are an expert information-extraction system. "
        "You always answer with a single JSON object matching the requested schema "
        "and nothing else."
    )

    USER_PROMPT_TEMPLATE = (
        "Extract named entities from the news headline below.\n\n"
        "Fill the JSON schema with these four keys, each a list of strings:\n"
        '- "persons": names of individual people.\n'
        '- "organizations": companies, agencies, sports teams, government bodies.\n'
        '- "locations": countries, cities, regions, landmarks.\n'
        '- "events": named events (e.g. \'World Cup\', \'Olympics\', \'Summit of the Americas\').\n\n'
        "Rules:\n"
        "- If a category has no entities, use [].\n"
        "- Do not put an entity in more than one category.\n"
        "- Do not invent entities that are not literally in the headline.\n"
        "- Keep the exact surface form (casing/spelling) from the headline.\n\n"
        "Headline: {text}"
    )

    def __init__(
        self,
        api_key: str | None = None,
        model: str = "gemini-3-flash-preview",
        retries: int = 3,
        rewrite_cache: bool = False,
        cache_path: str = ".cache",
        base_url: str = "https://openrouter.ai/api/v1",
        referer: str | None = None,
        app_title: str = "course-dl-annotation",
    ):
        api_key = api_key or os.environ.get("OPENROUTER_API_KEY")
        assert api_key and api_key != "PASTE_YOUR_OPENROUTER_KEY_HERE", (
            "Please set OPENROUTER_API_KEY (see the setup cell at the top of the notebook)"
        )
        self._api_key = api_key
        self._local = threading.local()
        self.model = model
        self.retries = retries
        self.rewrite_cache = rewrite_cache
        self.cache_path = cache_path
        self.base_url = base_url
        self.referer = referer
        self.app_title = app_title

        os.makedirs(cache_path, exist_ok=True)
        full_cache_path = os.path.join(cache_path, "openrouter_annotate_cache.diskcache")
        cache_settings = dc.DEFAULT_SETTINGS.copy()
        cache_settings["eviction_policy"] = "none"
        cache_settings["size_limit"] = int(1e12)
        cache_settings["cull_limit"] = 0
        self._cache = dc.Cache(full_cache_path, **cache_settings)

    def _client(self):
        if not hasattr(self._local, "client"):
            default_headers = {}
            if self.referer:
                default_headers["HTTP-Referer"] = self.referer
            if self.app_title:
                default_headers["X-Title"] = self.app_title

            self._local.client = OpenAI(
                api_key=self._api_key,
                base_url=self.base_url,
                default_headers=default_headers or None,
            )
        return self._local.client

    def close(self) -> None:
        self._cache.close()

    def annotate(self, text: str) -> dict:
        """One headline -> entity dict; diskcache key ``(model, prompt)``."""
        prompt = self.USER_PROMPT_TEMPLATE.format(text=text)
        cache_key = (self.model, prompt)

        if cache_key in self._cache and not self.rewrite_cache:
            return dict(self._cache[cache_key])

        last_err = None
        for _ in range(self.retries):
            try:
                resp = self._client().chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": self.SYSTEM_PROMPT},
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0.0,
                    response_format={"type": "json_object"},
                )
                raw = (resp.choices[0].message.content or "{}").strip()
                data = Entities.model_validate(json.loads(raw)).model_dump()

                out = {}
                for k in ("persons", "organizations", "locations", "events"):
                    vals = data.get(k, []) or []
                    out[k] = sorted({str(v).strip() for v in vals if str(v).strip()})
                self._cache[cache_key] = out
                return out

            except Exception as e:
                last_err = e
                log.info("OpenRouter annotate failed: %s", e)

        raise last_err

    def annotate_many(
        self,
        texts: list[str],
        max_workers: int = 4,
        desc: str = "Annotating",
    ) -> list[dict]:
        """Entity dicts in input order; uses a thread pool when ``max_workers > 1``."""
        if max_workers <= 1:
            return [self.annotate(t) for t in tqdm(texts, desc=desc)]

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            return list(tqdm(ex.map(self.annotate, texts), total=len(texts), desc=desc))

    def annotate_corpus(
        self,
        texts: list[str],
        max_workers: int = 8,
        desc: str = "Annotating",
    ) -> list[dict]:
        """Build ``[{text, entities}, ...]``; skips failures (same behaviour as a simple loop)."""

        def _one(t: str):
            try:
                return {"text": t, "entities": self.annotate(t)}
            except Exception as e:
                print("  skipped:", t[:70], "->", e)
                return None

        if max_workers <= 1:
            out = []
            for t in tqdm(texts, desc=desc):
                r = _one(t)
                if r is not None:
                    out.append(r)
            return out

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            results = list(tqdm(ex.map(_one, texts), total=len(texts), desc=desc))
        return [r for r in results if r is not None]


annotator = Annotator()

In [8]:
# Quick sanity check on one headline.
example = headlines[0]
print("Headline:", example)
print("Annotation:", annotator.annotate(example))

Headline: Policeman 'saw fatal train crash' An off-duty policeman watched a train plough into a car on a level crossing  in Berkshire, killing six people.
Annotation: {'persons': [], 'organizations': [], 'locations': ['Berkshire'], 'events': []}


In [9]:
# Annotate the whole corpus in parallel (threads). Results are cached on disk (Annotator._cache).
# We still export JSONL for a portable snapshot and for the rest of the notebook.
Path("data").mkdir(exist_ok=True)
export_path = Path("data/annotations.jsonl")

MAX_ANNOTATION_WORKERS = 8  # tune for API rate limits
annotations = annotator.annotate_corpus(headlines, max_workers=MAX_ANNOTATION_WORKERS)

with export_path.open("w") as f:
    for a in annotations:
        f.write(json.dumps(a, ensure_ascii=False) + "\n")

print(f"Wrote {len(annotations)} annotations to {export_path} (diskcache + parallel threads)")

Annotating: 100%|██████████| 400/400 [01:17<00:00,  5.14it/s]


Wrote 400 annotations to data/annotations.jsonl (diskcache + parallel threads)


In [10]:
# Eyeball a handful — always do this!
for a in annotations[:6]:
    print("•", a["text"])
    print("  ", a["entities"])
    print()

• Policeman 'saw fatal train crash' An off-duty policeman watched a train plough into a car on a level crossing  in Berkshire, killing six people.
   {'persons': [], 'organizations': [], 'locations': ['Berkshire'], 'events': []}

• Silver finale for USA In the last event of the 2004 Olympic Games, the United States track team produced one last surprise. Meb Keflezighi, a native of Eritrea who moved to the United States as
   {'persons': ['Meb Keflezighi'], 'organizations': ['USA', 'United States track team'], 'locations': ['Eritrea', 'United States'], 'events': ['2004 Olympic Games']}

• Polish Hostage Freed in Iraq Already in Warsaw  WARSAW (Reuters) - A Polish woman kidnapped in Iraq last  month has been freed and flown to Poland and said she was  treated well, raising hopes for other foreign hostages.
   {'persons': [], 'organizations': ['Reuters'], 'locations': ['Iraq', 'Poland', 'WARSAW', 'Warsaw'], 'events': []}

• Wall Street Journal to Start Saturday Issue he Wall Street Journa

## 3. Build the student dataset

We format each annotated example as a short chat turn:

* **system** – the task description,
* **user** – the headline,
* **assistant** – the JSON annotation that Gemini produced (this is what the student learns to imitate).

We then split 80 / 10 / 10 into train / val / test.

In [12]:
print("Splitting into train/val/test")
rng = random.Random(0)
shuffled = annotations[:]
rng.shuffle(shuffled)
n = len(shuffled)
train = shuffled[: int(0.7 * n)]
val   = shuffled[int(0.7 * n): int(0.8 * n)]
test  = shuffled[int(0.8 * n):]
print(f"train/val/test = {len(train)}/{len(val)}/{len(test)}")

Splitting into train/val/test
train/val/test = 280/40/80


In [13]:
# --- Student backbone (non-thinking regime for §§3-5) ---
STUDENT_VARIANT = "qwen3_nothink"

STUDENT_MODEL_IDS = {
    "qwen3_nothink": "Qwen/Qwen3-0.6B",
    "qwen3_think": "Qwen/Qwen3-0.6B",  # used later in optional §9
}
STUDENT_NAME = STUDENT_MODEL_IDS[STUDENT_VARIANT]
QWEN3_ENABLE_THINKING = False


def is_qwen3_student() -> bool:
    return True


def student_chat_template_kwargs() -> dict:
    # Sections 3-5 are fixed to non-thinking.
    return {"enable_thinking": QWEN3_ENABLE_THINKING}


def strip_qwen3_thinking_output(text: str) -> str:
    """If thinking mode is enabled later (optional §9), keep only content after </think>."""
    s = text.strip()
    for marker in ("</think>", "`</think>`"):
        if marker in s:
            s = s.split(marker)[-1].strip()
    return s


def student_run_tag() -> str:
    return STUDENT_VARIANT


STUDENT_ARTIFACT_DIR = "out_student_qwen3_nothink"

STUDENT_SYSTEM = (
    "You extract named entities from news headlines and return strict JSON "
    "with keys 'persons', 'organizations', 'locations', 'events'. "
    "Each value is a list of strings copied verbatim from the headline."
)
STUDENT_USER_TPL = "Headline: {text}"

def to_messages(example):
    row = {
        "messages": [
            {"role": "system",    "content": STUDENT_SYSTEM},
            {"role": "user",      "content": STUDENT_USER_TPL.format(text=example["text"])},
            {"role": "assistant", "content": json.dumps(example["entities"], ensure_ascii=False)},
        ],
    }
    if k := student_chat_template_kwargs():
        row["chat_template_kwargs"] = k
    return row

train_ds = Dataset.from_list([to_messages(e) for e in train])
val_ds   = Dataset.from_list([to_messages(e) for e in val])
print(train_ds)
print("\nExample messages:")
for m in train_ds[0]["messages"]:
    print(f"  [{m['role']}] {m['content'][:120]}")


Dataset({
    features: ['messages', 'chat_template_kwargs'],
    num_rows: 280
})

Example messages:
  [system] You extract named entities from news headlines and return strict JSON with keys 'persons', 'organizations', 'locations',
  [user] Headline: Rethinking Big Pharma Risk is knocking Big Pharma stocks down. They won't be getting up anytime soon.
  [assistant] {"persons": [], "organizations": ["Big Pharma"], "locations": [], "events": []}


## 4. Load the student and measure its **zero-shot** baseline

This section uses **Qwen3 non-thinking** only (`enable_thinking=False`) to keep the main pipeline deterministic.

Before any fine-tuning we want to know how well the raw instruction-tuned student already solves the task. This is the number we have to beat.


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Sections 3-5 are fixed to Qwen3 non-thinking.
STUDENT_VARIANT = "qwen3_nothink"
STUDENT_MODEL_IDS = {
    "qwen3_nothink": "Qwen/Qwen3-0.6B",
    "qwen3_think": "Qwen/Qwen3-0.6B",  # used only in optional §9
}
STUDENT_NAME = STUDENT_MODEL_IDS[STUDENT_VARIANT]
QWEN3_ENABLE_THINKING = False

if "student_chat_template_kwargs" not in globals():
    def is_qwen3_student() -> bool:
        return True

    def student_chat_template_kwargs() -> dict:
        return {"enable_thinking": QWEN3_ENABLE_THINKING}

    def strip_qwen3_thinking_output(text: str) -> str:
        s = text.strip()
        for marker in ("</think>", "`</think>`"):
            if marker in s:
                s = s.split(marker)[-1].strip()
                
        return s

    def student_run_tag() -> str:
        return STUDENT_VARIANT

STUDENT_ARTIFACT_DIR = "out_student_qwen3_nothink"

tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    dtype=dtype,          # transformers v5: `torch_dtype` was renamed to `dtype`
    device_map=DEVICE,
)
print(f"Student variant: {STUDENT_VARIANT} → {STUDENT_NAME}")
print(f"Qwen3 enable_thinking (tokenizer): {QWEN3_ENABLE_THINKING}")
print(f"Params : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 2158.65it/s]


Student variant: qwen3_nothink → Qwen/Qwen3-0.6B
Qwen3 enable_thinking (tokenizer): False
Params : 596.0 M


In [19]:
# Prompt-only example on the base student (no fine-tuning, no LoRA adapter).
sample_text = "Apple opens a new AI research center in Zurich with ETH collaboration."

messages = [
    {"role": "system", "content": STUDENT_SYSTEM},
    {"role": "user", "content": STUDENT_USER_TPL.format(text=sample_text)},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    **student_chat_template_kwargs(),
).to(model.device)

prompt_len = inputs["input_ids"].shape[1]
out = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

raw_baseline = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()

print("Input:", sample_text)
print("\nRaw model output:\n", raw_baseline)

Input: Apple opens a new AI research center in Zurich with ETH collaboration.

Raw model output:
 ```json
{
  "persons": [],
  "organizations": ["ETH Zurich"],
  "locations": ["Zurich"],
  "events": ["Apple opens a new AI research center in Zurich with ETH collaboration."]
}
```


In [20]:
@torch.no_grad()
def generate_json(text: str, max_new_tokens: int | None = None) -> str:
    if max_new_tokens is None:
        max_new_tokens = 512 if (is_qwen3_student() and QWEN3_ENABLE_THINKING) else 200

    messages = [
        {"role": "system", "content": STUDENT_SYSTEM},
        {"role": "user",   "content": STUDENT_USER_TPL.format(text=text)},
    ]
    tmpl_kw = student_chat_template_kwargs()
    # transformers v5: apply_chat_template returns a BatchEncoding; Qwen3 passes enable_thinking via tmpl_kw.
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        **tmpl_kw,
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    if is_qwen3_student() and QWEN3_ENABLE_THINKING:
        gen = strip_qwen3_thinking_output(gen)
        
    return gen

def try_parse(s: str):
    s = s.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    m = re.search(r"\{.*\}", s, re.S)
    if not m:
        return None
        
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

    out = {}
    for k in ("persons", "organizations", "locations", "events"):
        vals = d.get(k, []) or []
        if not isinstance(vals, list):
            vals = [vals]
        out[k] = [str(v) for v in vals]
        
    return out

EMPTY = {"persons": [], "organizations": [], "locations": [], "events": []}

def entity_prf(pred, gold):
    keys = ("persons", "organizations", "locations", "events")
    P = {(k, v.lower().strip()) for k in keys for v in (pred.get(k) or [])}
    G = {(k, v.lower().strip()) for k in keys for v in (gold.get(k) or [])}
    if not P and not G:
        return 1.0, 1.0, 1.0
    tp = len(P & G)
    p = tp / len(P) if P else 0.0
    r = tp / len(G) if G else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f

def evaluate(examples, tag=""):
    ps, rs, fs, parseable = [], [], [], 0
    preds = []
    for ex in tqdm(examples, desc=f"eval[{tag}]"):
        raw = generate_json(ex["text"])
        pred = try_parse(raw)
        parseable += int(pred is not None)
        if pred is None:
            pred = dict(EMPTY)
        p, r, f = entity_prf(pred, ex["entities"])
        ps.append(p); rs.append(r); fs.append(f)
        preds.append(pred)
        
    print(
        f"[{tag}] parseable={parseable}/{len(examples)} "
        f" P={np.mean(ps):.3f}  R={np.mean(rs):.3f}  F1={np.mean(fs):.3f}"
    )
    return {"P": float(np.mean(ps)), "R": float(np.mean(rs)), "F1": float(np.mean(fs)), "preds": preds}

In [21]:
print("Evaluating the baseline model without fine-tuning")
model.eval()
baseline = evaluate(test, tag="baseline")

Evaluating the baseline model without fine-tuning


eval[baseline]: 100%|██████████| 80/80 [01:05<00:00,  1.22it/s]

[baseline] parseable=77/80  P=0.279  R=0.337  F1=0.278


## 5. Fine-tune with LoRA via `SFTTrainer`

We attach low-rank adapters to the attention and MLP projections of the student.
With `r=16` the adapter is only a few million extra parameters — the original 0.5 B weights remain frozen.

`SFTTrainer` understands the `messages` column out of the box: it applies the tokenizer's chat template and (by default) masks the loss so that the model is only trained on the **assistant** tokens. That's exactly what we want here.

This main training path is **non-thinking only** (`enable_thinking=False`).


In [22]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


print("Fine-tuning the student with LoRA on data annotated by teacher...")

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

if "STUDENT_ARTIFACT_DIR" not in globals():
    STUDENT_ARTIFACT_DIR = "out_student_qwen3_nothink"

sft_cfg = SFTConfig(
    output_dir=STUDENT_ARTIFACT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="no",
    eval_strategy="epoch",
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    max_length=512,
    packing=False,
    assistant_only_loss=True,   # TRL masks the system/user turns -> loss only on the JSON answer
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_cfg,
    processing_class=tokenizer,
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Could not find the bitsandbytes CUDA binary at PosixPath('/home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_cuda130.so')
Could not load bitsandbytes native library: /mnt/data/users/artemshelmanov/workspace/conda/bin/../lib/libstdc++.so.6: version `GLIBCXX_3.4.32' not found (required by /home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/bitsandbytes/libbitsandbytes_cpu.so)
Traceback (most recent call last):
  File "/home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 85, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 72, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/h

Epoch,Training Loss,Validation Loss
1,0.258085,0.188198
2,0.108756,0.143630
3,0.063385,0.145531


TrainOutput(global_step=54, training_loss=0.24721072862545648, metrics={'train_runtime': 47.6429, 'train_samples_per_second': 17.631, 'train_steps_per_second': 1.133, 'total_flos': 356908400640000.0, 'train_loss': 0.24721072862545648})

## 6. Evaluate the fine-tuned student

In [23]:
model = trainer.model  # the PEFT-wrapped model with trained LoRA adapters
model.eval()
finetuned = evaluate(test, tag="finetuned")
finetuned_nonthinking = finetuned

print("\n=== Test-set summary ===")
if "baseline" not in globals():
    raise RuntimeError("Run §4 before this cell.")
_zero = baseline
print(f"zero-shot : P={_zero['P']:.3f}  R={_zero['R']:.3f}  F1={_zero['F1']:.3f}")
print(f"finetuned : P={finetuned['P']:.3f}  R={finetuned['R']:.3f}  F1={finetuned['F1']:.3f}")
print(f"Δ F1      : {finetuned['F1'] - _zero['F1']:+.3f}")


eval[finetuned]: 100%|██████████| 80/80 [01:01<00:00,  1.29it/s]

[finetuned] parseable=79/80  P=0.776  R=0.726  F1=0.736

=== Test-set summary ===
zero-shot : P=0.279  R=0.337  F1=0.278
finetuned : P=0.776  R=0.726  F1=0.736
Δ F1      : +0.458


In [24]:
# Prompt-only example on the base student (no fine-tuning, no LoRA adapter).
sample_text = "Apple opens a new AI research center in Zurich with ETH collaboration."

messages = [
    {"role": "system", "content": STUDENT_SYSTEM},
    {"role": "user", "content": STUDENT_USER_TPL.format(text=sample_text)},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    **student_chat_template_kwargs(),
).to(model.device)

prompt_len = inputs["input_ids"].shape[1]
out = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

raw_baseline = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()

print("Input:", sample_text)
print("\nRaw model output:\n", raw_baseline)

Input: Apple opens a new AI research center in Zurich with ETH collaboration.

Raw model output:
 {"persons": [], "organizations": ["Apple", "ETH"], "locations": ["Zurich"], "events": []}


## 7. Qualitative comparison

Let's look at the predictions side by side for a few test headlines.

In [25]:
for ex, pred in list(zip(test, finetuned["preds"]))[:6]:
    print("HEADLINE :", ex["text"])
    print("TEACHER  :", ex["entities"])
    print("STUDENT  :", pred)
    print("-" * 72)

HEADLINE : Day after Thanksgiving busy for yule tree farms Danville -- Liesel and Jeff Ray, of Avon, followed tradition Friday morning when they left the house. No, they did not head to the nearest mall, hoping to be the first in line for Christmas bargains.
TEACHER  : {'persons': ['Jeff Ray', 'Liesel'], 'organizations': [], 'locations': ['Avon', 'Danville'], 'events': ['Christmas', 'Thanksgiving']}
STUDENT  : {'persons': ['Jeff Ray', 'Liesel'], 'organizations': [], 'locations': ['Avon', 'Danville'], 'events': ['Thanksgiving', 'Christmas']}
------------------------------------------------------------------------
HEADLINE : Yes, but can your VoIP service do this? SAN FRANCISCO--The next battle between top Internet phone service providers Vonage and AT amp;T will involve features like voice mail, according to key executives at both companies.
TEACHER  : {'persons': [], 'organizations': ['AT amp;T', 'Vonage'], 'locations': ['SAN FRANCISCO'], 'events': []}
STUDENT  : {'persons': [], 'organ

## 8. Save the LoRA adapter

Only the tiny LoRA delta is saved (a few MB), not the whole 0.5 B model.

In [26]:
adapter_dir = f"{STUDENT_ARTIFACT_DIR}/lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Adapter saved to {adapter_dir}/")
import subprocess
subprocess.run(["du", "-sh", adapter_dir], check=False)

Adapter saved to out_student_qwen3_nothink/lora_adapter/
50M	out_student_qwen3_nothink/lora_adapter


CompletedProcess(args=['du', '-sh', 'out_student_qwen3_nothink/lora_adapter'], returncode=0)

## 9. Optional: Thinking regime and comparison vs non-thinking fine-tuned model

The main pipeline (§§3-8) trains and evaluates **Qwen3 non-thinking** only.

This optional section first computes a **baseline-only** comparison (no fine-tuning): `baseline` non-thinking vs `baseline_thinking`.

Then it runs a second training in **Qwen3 thinking** mode (`enable_thinking=True`) on the same train/val split, evaluates it on the same test set, and compares it directly against the already computed `finetuned_nonthinking` result.


In [19]:
# Optional: compare baseline (no fine-tuning) and then fine-tune Qwen3 in thinking mode.
# Prereqs: run §§3-8 first so `train`, `val`, `test`, `lora_cfg`, `sft_cfg`, `baseline`, and `finetuned_nonthinking` exist.
assert "finetuned_nonthinking" in globals(), "Run §§3-8 first (need non-thinking fine-tuned metrics)."
assert "baseline" in globals(), "Run §4 first (need non-thinking zero-shot baseline)."

# Preserve current non-thinking runtime objects.
_prev_variant = STUDENT_VARIANT
_prev_name = STUDENT_NAME
_prev_thinking = QWEN3_ENABLE_THINKING
_prev_artifact_dir = STUDENT_ARTIFACT_DIR
_prev_model = model
_prev_tokenizer = tokenizer

# Switch regime to thinking.
STUDENT_VARIANT = "qwen3_think"
STUDENT_NAME = STUDENT_MODEL_IDS[STUDENT_VARIANT]
QWEN3_ENABLE_THINKING = True
STUDENT_ARTIFACT_DIR = "out_student_qwen3_think"

# Rebuild chat datasets so chat_template_kwargs uses enable_thinking=True.
train_ds_thinking = Dataset.from_list([to_messages(e) for e in train])
val_ds_thinking = Dataset.from_list([to_messages(e) for e in val])

# Fresh base model/tokenizer for a clean second training run.
tokenizer_thinking = AutoTokenizer.from_pretrained(STUDENT_NAME)
if tokenizer_thinking.pad_token_id is None:
    tokenizer_thinking.pad_token = tokenizer_thinking.eos_token

model_thinking = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    dtype=dtype,
    device_map=DEVICE,
)

# Baseline comparison (no fine-tuning): non-thinking vs thinking.
model = model_thinking
tokenizer = tokenizer_thinking
model.eval()
baseline_thinking = evaluate(test, tag="baseline_thinking")

qwen3_baseline_mode_comparison = {
    "qwen3_nothink": baseline,
    "qwen3_think": baseline_thinking,
}

print("\n=== Baseline comparison (no fine-tuning) ===")
print(f"non-thinking : P={baseline['P']:.3f}  R={baseline['R']:.3f}  F1={baseline['F1']:.3f}")
print(f"thinking     : P={baseline_thinking['P']:.3f}  R={baseline_thinking['R']:.3f}  F1={baseline_thinking['F1']:.3f}")
print(f"Δ F1 (thinking - non-thinking): {baseline_thinking['F1'] - baseline['F1']:+.3f}")






eval[baseline_thinking]: 100%|██████████| 80/80 [06:49<00:00,  5.12s/it]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[baseline_thinking] parseable=59/80  P=0.289  R=0.413  F1=0.320

=== Baseline comparison (no fine-tuning) ===
non-thinking : P=0.264  R=0.356  F1=0.276
thinking     : P=0.289  R=0.413  F1=0.320
Δ F1 (thinking - non-thinking): +0.044


Tokenizing eval dataset: 100%|██████████| 40/40 [00:00<00:00, 2661.11 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,0.221998,0.176447
2,0.091910,0.147484
3,0.047526,0.145766


TrainOutput(global_step=54, training_loss=0.23066437244415283, metrics={'train_runtime': 44.0109, 'train_samples_per_second': 19.086, 'train_steps_per_second': 1.227, 'total_flos': 352236994560000.0, 'train_loss': 0.23066437244415283})

In [20]:
sft_cfg_thinking = SFTConfig(
    output_dir=STUDENT_ARTIFACT_DIR,
    num_train_epochs=sft_cfg.num_train_epochs,
    per_device_train_batch_size=sft_cfg.per_device_train_batch_size,
    gradient_accumulation_steps=sft_cfg.gradient_accumulation_steps,
    learning_rate=sft_cfg.learning_rate,
    lr_scheduler_type=sft_cfg.lr_scheduler_type,
    warmup_ratio=sft_cfg.warmup_ratio,
    logging_steps=sft_cfg.logging_steps,
    save_strategy=sft_cfg.save_strategy,
    eval_strategy=sft_cfg.eval_strategy,
    bf16=sft_cfg.bf16,
    fp16=sft_cfg.fp16,
    max_length=sft_cfg.max_length,
    packing=sft_cfg.packing,
    assistant_only_loss=sft_cfg.assistant_only_loss,
    report_to=sft_cfg.report_to,
    seed=sft_cfg.seed,
)

trainer_thinking = SFTTrainer(
    model=model_thinking,
    args=sft_cfg_thinking,
    train_dataset=train_ds_thinking,
    eval_dataset=val_ds_thinking,
    peft_config=lora_cfg,
    processing_class=tokenizer_thinking,
)
trainer_thinking.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/artemshelmanov/main/workspace/conda/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Tokenizing eval dataset: 100%|██████████| 40/40 [00:00<00:00, 2801.62 examples/s]


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Prompt-only LLM generation (no fine-tuning): run baseline student on a custom headline.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

sample_text = "Apple opens a new AI research center in Zurich with ETH collaboration."

baseline_tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME)
if baseline_tokenizer.pad_token_id is None:
    baseline_tokenizer.pad_token = baseline_tokenizer.eos_token

baseline_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
baseline_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_NAME,
    dtype=baseline_dtype,
    device_map=DEVICE,
).eval()

tmpl_kw = student_chat_template_kwargs() if "student_chat_template_kwargs" in globals() else {}
messages = [
    {"role": "system", "content": STUDENT_SYSTEM},
    {"role": "user", "content": STUDENT_USER_TPL.format(text=sample_text)},
]

inputs = baseline_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    **tmpl_kw,
).to(baseline_model.device)

prompt_len = inputs["input_ids"].shape[1]
out = baseline_model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=baseline_tokenizer.eos_token_id,
)

raw = baseline_tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
if "strip_qwen3_thinking_output" in globals() and globals().get("QWEN3_ENABLE_THINKING", False):
    raw = strip_qwen3_thinking_output(raw)

print("Input:", sample_text)
print("\nRaw model output:\n", raw)
if "try_parse" in globals():
    print("\nParsed JSON:\n", try_parse(raw))

In [ ]:
# Evaluate thinking fine-tuned model with existing evaluate()/generate_json helpers.
model = trainer_thinking.model
tokenizer = tokenizer_thinking
model.eval()
finetuned_thinking = evaluate(test, tag="finetuned_thinking")

print("\n=== Fine-tuned comparison: non-thinking vs thinking ===")
print(f"non-thinking : P={finetuned_nonthinking['P']:.3f}  R={finetuned_nonthinking['R']:.3f}  F1={finetuned_nonthinking['F1']:.3f}")
print(f"thinking     : P={finetuned_thinking['P']:.3f}  R={finetuned_thinking['R']:.3f}  F1={finetuned_thinking['F1']:.3f}")
print(f"Δ F1 (thinking - non-thinking): {finetuned_thinking['F1'] - finetuned_nonthinking['F1']:+.3f}")

# Restore non-thinking runtime objects for subsequent cells.
STUDENT_VARIANT = _prev_variant
STUDENT_NAME = _prev_name
QWEN3_ENABLE_THINKING = _prev_thinking
STUDENT_ARTIFACT_DIR = _prev_artifact_dir
model = _prev_model
tokenizer = _prev_tokenizer

eval[finetuned_thinking]:   1%|▏         | 1/80 [00:09<12:11,  9.26s/it]eval[finetuned_thinking]:   2%|▎         | 2/80 [00:19<12:33,  9.66s/it]eval[finetuned_thinking]:  49%|████▉     | 39/80 [06:20<06:23,  9.37s/it]